# Mount Drive + Set Up

In [4]:
from google.colab import drive
drive.mount('/content/drive')

BASE        = '/content/drive/MyDrive/rare_disease_project/data'
images_base = f'{BASE}/zebramap/images'

import os, pickle, numpy as np, pandas as pd, json
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

with open(f'{BASE}/config.pkl', 'rb') as f:
    config = pickle.load(f)

tier_a = [str(t) for t in config['tier_a']]
print(f"Tier A diseases : {len(tier_a)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device : cuda
GPU    : Tesla T4
Tier A diseases : 62


# Build Image DataFrame

In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import random, json

# ── Sample max 50 images per disease directly from Drive ──
print("Building dataset (50 images per disease max)...")

random.seed(42)
records = []

for disease in tier_a:
    disease_path = os.path.join(images_base, disease)
    if not os.path.exists(disease_path):
        continue

    imgs = []
    for root, dirs, files in os.walk(disease_path):
        for fname in files:
            if fname.lower().endswith(('.jpg','.png','.jpeg')):
                imgs.append({
                    'image_path': os.path.join(root, fname),
                    'orpha_code': disease,
                })

    # Max 50 per disease
    sampled = random.sample(imgs, min(50, len(imgs)))
    records.extend(sampled)

img_df = pd.DataFrame(records)
print(f"Total images   : {len(img_df):,}")
print(f"Unique diseases: {img_df['orpha_code'].nunique()}")
print(f"Avg per disease: {len(img_df)/img_df['orpha_code'].nunique():.0f}")

# Disease names
with open(f'{BASE}/zebramap/structured_cases.json', 'r') as f:
    cases = json.load(f)

disease_names = {}
for orpha, case_dict in cases.items():
    for case_id, case_data in case_dict.items():
        name = case_data['CaseData']['ActualDiagnosis'].get(
            'DiseaseName', '')
        if name:
            disease_names[str(orpha)] = name
            break

img_df['disease_name'] = img_df['orpha_code'].map(
    disease_names).fillna('Unknown')

# Label encode
le_img = LabelEncoder()
le_img.fit(img_df['orpha_code'])
img_df['label'] = le_img.transform(img_df['orpha_code'])
NUM_CLASSES     = len(le_img.classes_)

# Stratified split
train_df, test_df = train_test_split(
    img_df,
    test_size    = 0.2,
    random_state = 42,
    stratify     = img_df['orpha_code']
)

print(f"\nNum classes  : {NUM_CLASSES}")
print(f"Train images : {len(train_df):,}")
print(f"Test images  : {len(test_df):,}")

with open(f'{BASE}/label_encoder_image.pkl', 'wb') as f:
    pickle.dump(le_img, f)
print("✅ Label encoder saved")

Building dataset (50 images per disease max)...
Total images   : 3,100
Unique diseases: 62
Avg per disease: 50

Num classes  : 62
Train images : 2,480
Test images  : 620
✅ Label encoder saved


# Dataset + Transforms

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.1, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row['label'])
        try:
            img = Image.open(row['image_path']).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except:
            img = torch.zeros(3, 224, 224)
        return img, label

# Test
sample = ImageDataset(train_df.head(4), train_transforms)
img, lbl = sample[0]
print(f"✅ Dataset working")
print(f"   Shape : {img.shape}")
print(f"   Label : {lbl}")

✅ Dataset working
   Shape : torch.Size([3, 224, 224])
   Label : 39


# EfficientNet-B4 Model

In [7]:
import torch.nn as nn
import timm

class ImageClassifier(nn.Module):
    def __init__(self, num_classes, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4',
            pretrained  = True,
            num_classes = 0,
            global_pool = 'avg'
        )
        feat_dim = self.backbone.num_features  # 1792
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.backbone(x))

model   = ImageClassifier(NUM_CLASSES).to(device)
dummy   = torch.randn(2, 3, 224, 224).to(device)
out     = model(dummy)
print(f"✅ Model ready")
print(f"   Output shape : {out.shape}")
print(f"   Parameters   : {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

✅ Model ready
   Output shape : torch.Size([2, 62])
   Parameters   : 18,613,894


# Training Functions

In [8]:
from sklearn.metrics import f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(1)==labels).sum().item()
        total      += labels.size(0)
    return total_loss/len(loader), correct/total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs      = model(imgs)
            loss         = criterion(outputs, labels)
            total_loss  += loss.item()
            probs        = torch.softmax(outputs, dim=1)
            preds        = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds,
                        average='macro', zero_division=0)
    probs_np = np.array(all_probs)
    top3     = sum(
        all_labels[i] in np.argsort(probs_np[i])[-3:]
        for i in range(len(all_labels))
    ) / len(all_labels)

    return total_loss/len(loader), acc, f1, top3, \
           all_labels, all_preds, all_probs

print("✅ Training functions ready")

✅ Training functions ready


#  Train (The Main Cell)

In [9]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

BATCH_SIZE = 32
EPOCHS     = 15
best_f1    = 0
best_path  = f'{BASE}/image_model_best.pt'

# DataLoaders — reading from LOCAL SSD (fast)
train_dataset = ImageDataset(train_df, train_transforms)
test_dataset  = ImageDataset(test_df,  test_transforms)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                           shuffle=True,  num_workers=2,
                           pin_memory=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2,
                           pin_memory=True)

# Class weights
class_weights = compute_class_weight(
    'balanced',
    classes = np.unique(train_df['label'].values),
    y       = train_df['label'].values
)
criterion = nn.CrossEntropyLoss(
    weight=torch.FloatTensor(class_weights).to(device))

# Fresh model + optimizer
model     = ImageClassifier(NUM_CLASSES).to(device)
optimizer = AdamW([
    {'params': model.backbone.parameters(),   'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 1e-3}
], weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f"Train : {len(train_dataset):,} | Test : {len(test_dataset):,}")
print(f"Classes: {NUM_CLASSES} | Batch: {BATCH_SIZE} | Epochs: {EPOCHS}")
print(f"Batches/epoch: {len(train_loader)}")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(
        model, train_loader, optimizer, criterion, device)
    scheduler.step()
    vl_loss, vl_acc, vl_f1, vl_top3, \
        labs, preds, probs = eval_epoch(
        model, test_loader, criterion, device)

    flag = ''
    if vl_f1 > best_f1:
        best_f1 = vl_f1
        torch.save({
            'epoch'        : epoch,
            'model_state'  : model.state_dict(),
            'val_f1'       : vl_f1,
            'val_acc'      : vl_acc,
            'val_top3'     : vl_top3,
            'num_classes'  : NUM_CLASSES,
            'label_encoder': le_img,
        }, best_path)
        flag = ' ✅ saved'

    print(f"Ep {epoch:02d}/{EPOCHS} | "
          f"Tr Loss:{tr_loss:.3f} Acc:{tr_acc:.3f} | "
          f"Val Loss:{vl_loss:.3f} Acc:{vl_acc:.3f} "
          f"F1:{vl_f1:.3f} Top3:{vl_top3:.3f}{flag}")

print(f"\n🎉 Training complete — Best F1: {best_f1:.4f}")

Train : 2,480 | Test : 620
Classes: 62 | Batch: 32 | Epochs: 15
Batches/epoch: 78
Ep 01/15 | Tr Loss:4.120 Acc:0.022 | Val Loss:4.053 Acc:0.037 F1:0.014 Top3:0.124 ✅ saved
Ep 02/15 | Tr Loss:3.904 Acc:0.067 | Val Loss:3.847 Acc:0.087 F1:0.053 Top3:0.198 ✅ saved
Ep 03/15 | Tr Loss:3.640 Acc:0.104 | Val Loss:3.791 Acc:0.081 F1:0.060 Top3:0.210 ✅ saved
Ep 04/15 | Tr Loss:3.425 Acc:0.145 | Val Loss:3.795 Acc:0.100 F1:0.070 Top3:0.232 ✅ saved
Ep 05/15 | Tr Loss:3.203 Acc:0.188 | Val Loss:3.822 Acc:0.076 F1:0.064 Top3:0.226
Ep 06/15 | Tr Loss:3.028 Acc:0.222 | Val Loss:3.875 Acc:0.084 F1:0.066 Top3:0.226
Ep 07/15 | Tr Loss:2.856 Acc:0.256 | Val Loss:3.912 Acc:0.095 F1:0.081 Top3:0.239 ✅ saved
Ep 08/15 | Tr Loss:2.657 Acc:0.303 | Val Loss:3.961 Acc:0.113 F1:0.096 Top3:0.247 ✅ saved
Ep 09/15 | Tr Loss:2.516 Acc:0.332 | Val Loss:4.049 Acc:0.116 F1:0.098 Top3:0.226 ✅ saved
Ep 10/15 | Tr Loss:2.366 Acc:0.383 | Val Loss:4.061 Acc:0.106 F1:0.093 Top3:0.239
Ep 11/15 | Tr Loss:2.271 Acc:0.406 | Val L

# Final Evaluation + Save Results

In [12]:
from sklearn.metrics import matthews_corrcoef
import pickle

checkpoint = torch.load(
    best_path,
    map_location=device,
    weights_only=False
)
model.load_state_dict(checkpoint['model_state'])

_, acc, f1, top3, labels, preds, probs = eval_epoch(
    model, test_loader, criterion, device)

probs_np = np.array(probs)
top5     = sum(
    labels[i] in np.argsort(probs_np[i])[-5:]
    for i in range(len(labels))
) / len(labels)
mcc = matthews_corrcoef(labels, preds)

print("=" * 55)
print("IMAGE MODEL — EXPERIMENT 3 RESULTS")
print("=" * 55)
print(f"Accuracy   : {acc:.4f}  ({acc*100:.2f}%)")
print(f"Macro F1   : {f1:.4f}")
print(f"Top-3 Acc  : {top3:.4f}  ({top3*100:.2f}%)")
print(f"Top-5 Acc  : {top5:.4f}  ({top5*100:.2f}%)")
print(f"MCC        : {mcc:.4f}")
print(f"Classes    : {NUM_CLASSES}")
print(f"Test images: {len(labels):,}")
print("=" * 55)

# Save results
results = {
    'experiment'      : 'exp3_image_tier_a_50perclass',
    'accuracy'        : acc,
    'macro_f1'        : f1,
    'top3_acc'        : top3,
    'top5_acc'        : top5,
    'mcc'             : mcc,
    'num_classes'     : NUM_CLASSES,
    'test_samples'    : len(labels),
    'train_samples'   : len(train_df),
    'images_per_class': 50,
}
with open(f'{BASE}/exp3_image_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print("✅ Results saved to Drive")
print(f"✅ Model saved: {best_path}")

# Compare with symptoms model
print("\n📊 COMPARISON SO FAR")
print("=" * 55)
print(f"{'Model':25} {'Acc':8} {'F1':8} {'Top-3':8}")
print("-" * 55)
print(f"{'Symptoms (Exp3)':25} {'34.73%':8} {'0.3463':8} {'54.76%':8}")
print(f"{'Image (Exp3)':25} {f'{acc*100:.2f}%':8} {f'{f1:.4f}':8} {f'{top3*100:.2f}%':8}")
print("=" * 55)

IMAGE MODEL — EXPERIMENT 3 RESULTS
Accuracy   : 0.1210  (12.10%)
Macro F1   : 0.1098
Top-3 Acc  : 0.2500  (25.00%)
Top-5 Acc  : 0.3210  (32.10%)
MCC        : 0.1068
Classes    : 62
Test images: 620
✅ Results saved to Drive
✅ Model saved: /content/drive/MyDrive/rare_disease_project/data/image_model_best.pt

📊 COMPARISON SO FAR
Model                     Acc      F1       Top-3   
-------------------------------------------------------
Symptoms (Exp3)           34.73%   0.3463   54.76%  
Image (Exp3)              12.10%   0.1098   25.00%  
